# CloudPath AI – AI-Based Cloud Movement Tracking and Rainfall Prediction System
## Phase 3: Satellite Image Analysis & Cloud Movement Tracking

**Context:** Ingests sequential meteorological satellite imagery (ISRO MOSDAC INSAT-3D / INSAT-3DR TIR1/VIS/WV and scientific HDF5 files) to:
1. Load and parse satellite data with automatic timestamp extraction
2. Preprocess images (resizing, CLAHE contrast enhancement, edge-preserving bilateral filtering)
3. Segment clouds and categorize vertical density (Thin, Medium, Deep Convective)
4. Track multiple cloud formations across sequential frames with unique IDs
5. Calculate cloud displacement vectors, compass headings, and velocities (px/h and km/h)
6. Extrapolate future cloud positions (+30 min, +1 hour, +2 hours) and project collision trajectories with Bengaluru
7. Analyze total cloud coverage dynamics (expansion vs. dissipation trends)

### 1. Library Imports & Setup

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

# Import modular CloudPath AI Phase 3 components
from satellite_loader import load_satellite_sequence, generate_synthetic_insat_sequence
from image_preprocessing import preprocess_satellite_image
from cloud_detection import detect_clouds
from cloud_tracking import MultiCloudTracker
from cloud_motion import calculate_cloud_motion, compute_frame_motion_summary
from cloud_prediction import predict_future_cloud_positions
from cloud_visualization import (
    save_detection_visualization,
    save_tracking_and_motion_visualization,
    save_prediction_extrapolation_visualization,
    save_coverage_trend_visualization
)

print('All Phase 3 satellite tracking modules imported successfully!')

### 2. Ingesting & Inspecting Satellite Sequence
Loads timestamped satellite observations (PNG, JPG, TIFF, HDF5) and sorts them chronologically.

In [ ]:
INPUT_DIR = 'satellite_input'
frames = load_satellite_sequence(INPUT_DIR)

print(f'Total Frames Loaded: {len(frames)}')
for i, fr in enumerate(frames, 1):
    ts_str = fr.timestamp.strftime('%Y-%m-%d %H:%M') if fr.timestamp else 'N/A'
    status = 'SIMULATED TEST DATA' if fr.is_synthetic else 'ACTUAL MOSDAC'
    print(f'  Frame {i}: {fr.filename} | Time: {ts_str} | Res: {fr.width}x{fr.height} | Band: {fr.band} ({status})')

### 3. Image Preprocessing & Cloud Density Segmentation
Applies CLAHE contrast enhancement, bilateral filtering, and multi-level density classification.

In [ ]:
sample_frame = frames[0]
preprocessed, prep_log = preprocess_satellite_image(sample_frame.image, target_size=(600, 600))
det_result = detect_clouds(preprocessed, method='otsu', band_type=sample_frame.band)

print(f'Preprocessed Image Shape : {preprocessed.shape}')
print(f'Cloud Coverage           : {det_result.cloud_coverage_pct:.2f}%')
print(f'Total Cloud Formations   : {det_result.metadata["num_cloud_clusters"]}')
print(f'  - Thin / Low Clouds    : {det_result.metadata["thin_cloud_pct"]}%')
print(f'  - Medium Clouds        : {det_result.metadata["medium_cloud_pct"]}%')
print(f'  - Deep Convective Tops : {det_result.metadata["dense_cloud_pct"]}%')

# Plot side-by-side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(sample_frame.image, cmap='gray'); axes[0].set_title('1. Raw Image'); axes[0].axis('off')
axes[1].imshow(preprocessed, cmap='magma'); axes[1].set_title('2. Enhanced (CLAHE)'); axes[1].axis('off')
axes[2].imshow(det_result.density_mask, cmap='viridis'); axes[2].set_title('3. Density Classification'); axes[2].axis('off')
plt.tight_layout()
plt.show()

### 4. Sequential Multi-Cloud Tracking & Velocity Vectors
Tracks centroids across consecutive frames, assigns persistent IDs, and calculates velocities.

In [ ]:
tracker = MultiCloudTracker(max_match_distance=130.0, min_area=250.0)
km_per_pixel = 4.0  # INSAT-3D nominal resolution
all_motion_vectors = []

for f_idx, fr in enumerate(frames):
    prep, _ = preprocess_satellite_image(fr.image, target_size=(600, 600))
    det = detect_clouds(prep, band_type=fr.band)
    active_clouds = tracker.update_tracks(det.binary_mask, prep, f_idx, fr.timestamp)
    
    # Compute motion for clouds with >= 2 observations
    for c in active_clouds:
        if len(c.history_centroids) >= 2:
            mv = calculate_cloud_motion(
                prev_centroid=c.history_centroids[-2],
                curr_centroid=c.history_centroids[-1],
                prev_time=c.history_timestamps[-2],
                curr_time=c.history_timestamps[-1],
                cloud_id=c.cloud_id,
                km_per_pixel=km_per_pixel
            )
            all_motion_vectors.append(mv)

summary = compute_frame_motion_summary(all_motion_vectors)
print(f'Tracking Summary across {len(frames)} frames:')
print(f'  Total Moving Clouds Tracked : {summary["total_tracked_clouds"]}')
print(f'  Mean Velocity (Pixels)      : {summary["mean_speed_px_hr"]:.1f} px/hour')
print(f'  Mean Velocity (Calibrated)  : {summary["mean_speed_kmh"]:.1f} km/h')
print(f'  Predominant Compass Heading : {summary["predominant_direction"]}')

### 5. Future Cloud Position Extrapolation (+30m, +1h, +2h)
Linearly extrapolates future cloud centroids with expanding uncertainty circles and collision alerts.

In [ ]:
target_bengaluru = (320.0, 360.0)
last_frame = frames[-1]
prep_last, _ = preprocess_satellite_image(last_frame.image, target_size=(600, 600))
det_last = detect_clouds(prep_last)
active_clouds = tracker.active_tracks.values()

print('Future Extrapolations for Active Cloud Clusters:')
for c in active_clouds:
    if len(c.history_centroids) >= 2:
        dx = c.history_centroids[-1][0] - c.history_centroids[-2][0]
        dy = c.history_centroids[-1][1] - c.history_centroids[-2][1]
        dt_hr = 0.5
        preds = predict_future_cloud_positions(
            cloud_id=c.cloud_id,
            curr_centroid=c.centroid,
            curr_bbox=c.bbox,
            dx_per_hour=dx/dt_hr,
            dy_per_hour=dy/dt_hr,
            current_time=last_frame.timestamp,
            lead_times_minutes=[30, 60, 120],
            target_station_coord=target_bengaluru
        )
        for p in preds:
            intersect_flag = '--> [ALERT: COLLISION/INTERSECTING]' if p.will_intersect_target else ''
            print(f'  Cloud {c.cloud_id} | +{p.lead_time_minutes} min | Pos: ({p.predicted_centroid[0]:.1f}, {p.predicted_centroid[1]:.1f}) | Uncertainty: +/-{p.uncertainty_radius_px:.1f}px {intersect_flag}')

### 6. Review Generated Artifacts & Reports
Displaying generated metrics from `satellite_tracking_outputs/`.

In [ ]:
metrics_df = pd.read_csv('satellite_tracking_outputs/satellite_tracking_metrics.csv')
print('First 5 records from satellite_tracking_metrics.csv:')
display(metrics_df.head())

with open('satellite_tracking_outputs/cloud_coverage_report.txt', 'r') as f:
    print('\nTechnical Report Summary:')
    print(f.read())